# Shared evaluation protocol

Shared evaluation protocol for every dataset folder (same rules as the NHANES paper).

evaluate(X, y) -> per-model ROC-AUC from RepeatedStratifiedKFold (5 folds x 3 repeats), preprocessing fitted inside folds.
Models: logistic regression (C=1), random forest (500 trees, min leaf 5), XGBoost (300 trees, depth 3, lr 0.05).
Leakage guard: drop_toxo_columns() removes every column that measures Toxoplasma itself (IgM, avidity, titre, OD,
index, rapid test, PCR, 'infected' flags) so only exposures / demographics / other biomarkers remain as inputs.

Loaded by every folder notebook with `%run -i datasets/common/protocol.ipynb`.

In [ ]:
import re
import warnings
import numpy as np
import pandas as pd
from pandas.api.types import is_numeric_dtype
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
warnings.filterwarnings('ignore')
MIN_CLASS = 20          # need at least this many positives AND negatives to evaluate
TOXO_PAT = re.compile(r'toxo|gondii|\btg\b|tg[-_ ]?ig|igm|ig[-_ ]?m\b|avid|titre|titer|\bod\b|optical|elisa|eclia|cmia|'
                      r'\bmat\b|\bifat?\b|\bifi\b|\bdat\b|dye|rdt|rapid|pcr|\bdna\b|serostat|seropos|seroneg|serolog|'
                      r'antibod|\bigg\b|ig[-_ ]?g\b|infect(ed|ion)?\b|positiv|negativ|result|\bpos\b|\bneg\b|s/co|\bu/ml|iu/ml|'
                      r'congenital|chorioret|retinochor|ocular toxo|toxoplasmosis', re.I)
ID_PAT = re.compile(r'^(id|no\.?|n°|nr|number|code|serial|sl\.?\s*no|s/n|sample|sample id|patient id|subject|'
                    r'record|case no|lab no|study id|participant|unnamed.*)$', re.I)

**`drop_toxo_columns`** — Drop columns that measure Toxoplasma (leakage) and identifier columns.

In [ ]:
def drop_toxo_columns(df, keep=()):
    """Drop columns that measure Toxoplasma (leakage) and identifier columns."""
    drop = [c for c in df.columns if c not in keep and (TOXO_PAT.search(str(c)) or ID_PAT.match(str(c).strip()))]
    return df.drop(columns=drop), drop

**`clean_features`** — Coerce numeric-looking text to numbers, drop near-empty / constant / free-text columns.

In [ ]:
def clean_features(X, max_levels=30, min_nonmissing=0.5):
    """Coerce numeric-looking text to numbers, drop near-empty / constant / free-text columns."""
    X = X.copy()
    X.columns = [str(c).strip() for c in X.columns]
    X = X.loc[:, ~X.columns.duplicated()]
    keep = []
    for c in X.columns:
        s = X[c]
        if not is_numeric_dtype(s):
            s = s.astype(str).str.strip().replace({'': np.nan, 'nan': np.nan, 'NaN': np.nan, 'None': np.nan, '.': np.nan, '-': np.nan})
            num = pd.to_numeric(s.str.replace(',', '.', regex=False), errors='coerce')
            if num.notna().sum() >= 0.9 * s.notna().sum() and s.notna().sum() > 0:
                s = num
            X[c] = s
        if X[c].notna().mean() < min_nonmissing or X[c].nunique(dropna=True) < 2:
            continue
        if not is_numeric_dtype(X[c]) and X[c].nunique(dropna=True) > max_levels:
            continue                      # free text / dates
        keep.append(c)
    return X[keep]

**`_prep`**

In [ ]:
def _prep(X):
    cat = [c for c in X.columns if not is_numeric_dtype(X[c])]
    num = [c for c in X.columns if c not in cat]
    return ColumnTransformer([
        ('c', Pipeline([('i', SimpleImputer(strategy='most_frequent')),
                        ('o', OneHotEncoder(handle_unknown='ignore', min_frequency=5, sparse_output=False))]), cat),
        ('n', Pipeline([('i', SimpleImputer(strategy='median', add_indicator=True)), ('s', StandardScaler())]), num)])

**`models`**

In [ ]:
def models(seed=42):
    from xgboost import XGBClassifier
    return {'LogReg': LogisticRegression(C=1.0, max_iter=5000),
            'RandomForest': RandomForestClassifier(n_estimators=500, min_samples_leaf=5, n_jobs=-1, random_state=seed),
            'XGBoost': XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, subsample=0.8, colsample_bytree=0.8,
                                     eval_metric='logloss', n_jobs=-1, random_state=seed)}

**`evaluate`** — Returns dict model -> (mean ROC-AUC, SD) over 5-fold x n_repeats CV, or None if too few cases.

In [ ]:
def evaluate(X, y, seed=42, n_repeats=3):
    """Returns dict model -> (mean ROC-AUC, SD) over 5-fold x n_repeats CV, or None if too few cases."""
    y = np.asarray(y).astype(int)
    if min(y.sum(), len(y) - y.sum()) < MIN_CLASS or X.shape[1] == 0:
        return None
    X = X.reset_index(drop=True)
    for c in X.columns:
        if not is_numeric_dtype(X[c]):
            X[c] = X[c].astype(object).where(X[c].notna(), np.nan).astype(object)
            X[c] = X[c].map(lambda v: v if v is np.nan or v is None else str(v))
    out = {}
    cv = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=n_repeats, random_state=seed).split(X, y))
    for name, m in models(seed).items():
        s = []
        for tr, va in cv:
            p = make_pipeline(_prep(X), m).fit(X.iloc[tr], y[tr])
            s.append(roc_auc_score(y[va], p.predict_proba(X.iloc[va])[:, 1]))
        out[name] = (float(np.mean(s)), float(np.std(s)))
    return out

**`summarise`**

In [ ]:
def summarise(key, X, y, res, extra=None):
    row = dict(dataset=key, n=len(y), positives=int(np.sum(y)), prevalence=round(float(np.mean(y)), 3), n_features=X.shape[1])
    if res:
        for m, (a, s) in res.items():
            row[f'{m}_auc'] = round(a, 3); row[f'{m}_sd'] = round(s, 3)
        row['best_auc'] = round(max(a for a, _ in res.values()), 3)
        row['best_model'] = max(res, key=lambda k: res[k][0])
    else:
        row['best_auc'] = np.nan
        row['best_model'] = ('not evaluated (no usable inputs)' if X.shape[1] == 0 else
                             f'not evaluated (fewer than {MIN_CLASS} positives or negatives)')
    row.update(extra or {})
    return row